# D50 Docker

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.10

# 컨테이너와 이미지 빌드: 도커 기초 개념부터 다중 컨테이너 서빙까지

**도커(Docker)는 애플리케이션 코드와 파이썬 런타임, 시스템 라이브러리, 설정 파일을 하나의 패키지로 묶어 어느 컴퓨터에서나 동일한 환경으로 실행해 주는 컨테이너 기반 가상화 플랫폼입니다.** 46~49일차에서 만든 FastAPI 추론 서버와 Streamlit 화면을 사내 서버로 옮길 때 코드와 `requirements.txt`만 전달하면 파이썬 버전이나 운영체제 차이로 실행이 깨지기 쉽습니다.

이번 차시에서는 도커의 기본 개념(이미지, 컨테이너, 도커 허브)과 현업 LLM 서빙 유즈케이스를 익히고, 운영체제별(**Windows WSL2 + Ubuntu**, **macOS**, **Linux**) 설치 및 필수 명령어(`pull`, `run`, `ps`, `logs`, `exec`, `rm`, `rmi`)를 실습한 뒤, FastAPI 서버와 Streamlit 화면을 `Dockerfile`과 `compose.yaml`로 빌드·조립합니다.

**이번 차시 학습목표**
1. 도커의 3대 핵심 요소(이미지, 컨테이너, 도커 허브)와 가상머신(VM) 대비 컨테이너의 구조적 장점을 설명할 수 있다.
2. 운영체제(Windows WSL2, macOS, Linux)에 도커 환경을 구성하고 7대 기본 명령어(`pull`, `run`, `ps`, `logs`, `exec`, `rm`, `rmi`)를 수행할 수 있다.
3. 레이어 캐시 최적화(`COPY requirements.txt` 선발 배치), 시크릿 분리(`.dockerignore`, 환경 변수), 포트 바인딩(`127.0.0.1`)을 적용해 FastAPI + Streamlit 서비스를 `Dockerfile`과 `compose.yaml`로 구성할 수 있다.

## 실행 환경

**실습은 로컬 PC(Windows WSL2, macOS, Linux) 터미널의 Docker CLI와 주피터 노트북(코랩 포함)을 병행합니다.** 노트북 코드 셀은 서빙용 FastAPI·Streamlit 코드와 `Dockerfile`, `compose.yaml`을 생성·검증하고, `docker` 명령어는 로컬 터미널에서 실행합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 내용 |
|---|---|
| 로컬 실습 환경 | **Windows**: WSL2(Ubuntu) + `docker-ce` CLI / **macOS**: `Colima` 또는 `Docker Desktop` / **Linux**: `docker-ce` CLI |
| 사용하는 이미지·포트 | `hello-world`, `nginx:latest`, `python:3.11-slim` / FastAPI `8000`번, Streamlit `8501`번 |

</div>

## 1. 도커란 무엇인가: 핵심 개념과 현업 LLM 서빙 유즈케이스

교재 : 6.1.2 2단계: LLM 컨테이너화하기

### 1.1 모든 준비물을 챙긴 '여행용 캐리어'와 `requirements.txt`의 한계

**도커 컨테이너는 옷·세면도구·충전기를 모두 챙긴 '여행용 캐리어'처럼 코드, 파이썬 인터프리터, OS 시스템 라이브러리, 설정 파일을 통째로 담아 어느 호텔(서버)에 가든 똑같은 조건으로 실행되게 합니다.**

`requirements.txt`는 파이썬 패키지 버전만 맞출 뿐 다음 세 가지는 고정하지 못합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 불일치 영역 | 내 개발 PC (예시) | 사내 운영 서버 (예시) | 발생하는 문제 |
|---|---|---|---|
| **파이썬 인터프리터 버전** | Python `3.11` | Ubuntu 기본 `3.10` / `3.12` | 특정 버전용 패키지(wheel) 설치 실패 및 문법·타입 호환성 오류 |
| **OS C 라이브러리 (`glibc`, `CUDA`)** | Ubuntu `24.04` / macOS | 구형 CentOS / Ubuntu `20.04` | PyTorch, NumPy, OpenCV, 토크나이저의 C/C++ 확장 모듈 로드 실패 (`ImportError`) |
| **시스템 도구 및 폰트·로케일** | `ffmpeg`, 한글 폰트 설치됨 | 최소 설치(Minimal) 리눅스 | 오디오/이미지 전처리 실패, 시각화 한글 깨짐, 인코딩(`UTF-8`) 오류 |

</div>

### 1.2 도커의 4대 장점과 3대 기본 컨셉 (이미지 · 컨테이너 · 도커 허브)

**도커는 ① 이식성(개발·운영 환경 100% 일치), ② 빠른 배포와 확장성(1~2초 내 기동 및 Scale-out), ③ 분리된 격리 환경(한 서버에서 여러 모델을 충돌 없이 실행), ④ 버전 관리와 손쉬운 공유라는 네 가지 장점을 제공합니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 3대 기본 컨셉 | 쉬운 비유 | 역할과 특징 |
|---|---|---|
| **1. 이미지 (Image)** | 붕어빵 틀 / 프로그램 설치 파일 | 실행에 필요한 파일·라이브러리·설정을 굳혀 둔 **읽기 전용(Read-Only) 템플릿**입니다. 한 번 만든 이미지는 변하지 않습니다. |
| **2. 컨테이너 (Container)** | 찍어낸 붕어빵 / 실행 중인 프로세스 | 이미지 위에 얇은 **쓰기 레이어**를 얹어 실제로 메모리에 떠서 돌아가는 격리 인스턴스입니다. |
| **3. 도커 허브 (Docker Hub)** | 앱 스토어 / 코드의 GitHub | 이미지를 검색해 내려받거나(`pull`) 내가 만든 이미지를 올려 공유(`push`)하는 **중앙 레지스트리**입니다. |

</div>

### 1.3 가상머신(VM) vs 컨테이너 구조와 현업 LLM 엔지니어의 4대 유즈케이스

**가상머신(VM)은 하이퍼바이저 위에 수십 GB짜리 게스트 OS 전체를 새로 부팅하느라 수 분이 걸리지만, 컨테이너는 호스트 리눅스 커널을 공유하면서 프로세스만 격리해 수백 MB 크기로 1~2초 만에 기동됩니다.**

![가상머신(VM)은 무거운 게스트 OS를 각각 부팅하지만, 도커 컨테이너는 호스트 리눅스 커널을 공유해 1~2초 만에 경량으로 기동됩니다.](lms-img:0)

*가상머신(VM)은 무거운 게스트 OS를 각각 부팅하지만, 도커 컨테이너는 호스트 리눅스 커널을 공유해 1~2초 만에 경량으로 기동됩니다.*

![이미지는 하나이고 바뀌지 않으며 컨테이너마다 자기만의 쓰기 레이어가 추가되었다가 함께 사라집니다.](lms-img:0)

*이미지는 하나이고 바뀌지 않으며 컨테이너마다 자기만의 쓰기 레이어가 추가되었다가 함께 사라집니다.*

위 그림처럼 **이미지는 읽기 전용**이고, 컨테이너 실행 중에 만든 로그나 DB 파일은 컨테이너의 **얇은 쓰기 레이어**에 적혔다가 **컨테이너를 삭제(`docker rm`)하면 함께 사라집니다.** 따라서 영구히 남길 데이터(로그, 벡터DB, 대용량 모델 가중치)는 호스트 폴더를 **볼륨(`volumes`)**으로 연결해야 합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 현업 LLM/ML 유즈케이스 | 컨테이너가 없을 때의 문제 | 도커 컨테이너 도입 효과 |
|---|---|---|
| **1. 사내 폐쇄망(Private) 서버 이관** | 외부 인터넷(`pip install`)이 막힌 사내 서버에 의존성 수동 설치 실패 | 개발망에서 검증한 이미지를 반입해 설치 과정 없이 즉시 기동 |
| **2. 공식 서빙 엔진 도입 (`vLLM`, `Triton`, `Qdrant`)** | CUDA·C++ 컴파일 버전 충돌로 설치에 수일 소요 | 검증된 공식 이미지를 `docker run` 한 줄로 내려받아 즉시 실행 |
| **3. 마이크로서비스 분리 (교재 6.7.2)** | UI 문구만 고쳐도 무거운 PyTorch까지 통째로 재배포해야 함 | 추론 서버(`api`)와 화면(`ui`) 컨테이너를 분리해 독립 배포 |
| **4. 단일 GPU 서버 내 다중 모델 격리** | 모델마다 요구하는 `transformers`/`torch` 버전이 달라 동시 설치 불가 | 각 모델을 독립된 컨테이너에 담아 한 서버에서 충돌 없이 동시 운영 |

</div>

### 체크포인트
1. 도커의 3대 기본 컨셉인 **이미지**, **컨테이너**, **도커 허브**의 관계를 설명하십시오. (목표 1)
2. `requirements.txt`를 모두 고정해도 서버에서 실행이 깨질 수 있는 요소를 두 가지 드십시오. (목표 1)
3. 컨테이너 안에서 생성한 파일이 `docker rm` 후 사라지는 이유를 레이어 구조로 설명하십시오. (목표 1)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. **이미지**는 실행 환경을 굳혀 둔 읽기 전용 템플릿이고, **컨테이너**는 이미지를 바탕으로 실행 중인 격리 프로세스이며, **도커 허브**는 이미지를 저장·공유(`pull`/`push`)하는 원격 중앙 저장소입니다.
2. **파이썬 인터프리터 버전**과 **운영체제 C 라이브러리(`glibc`, CUDA 런타임) 및 OS 패키지(`ffmpeg`, 폰트)** 차이 때문입니다.
3. 이미지는 읽기 전용이고 실행 중 만든 파일은 컨테이너의 임시 **쓰기 레이어**에 저장되는데, 컨테이너 삭제 시 쓰기 레이어도 함께 소멸하기 때문입니다(영구 보존은 `volumes` 마운트 사용).

</details>

## 2. 운영체제별 도커 설치(Windows WSL2 · macOS · Linux)와 기본 명령어 실습

### 2.1 [Windows 사용자] WSL2(Ubuntu) 설치하기

**도커 컨테이너는 리눅스 커널 위에서 동작하므로, Windows 사용자는 관리자 PowerShell에서 `wsl --install`과 `wsl --set-default-version 2` 두 명령어로 정식 리눅스 커널(WSL2)과 Ubuntu를 먼저 설치합니다.** 여기서 `wsl --install`은 Windows의 리눅스 하위 시스템 기능과 기본 Ubuntu 배포판을 한 번에 내려받아 설치하는 명령어이고, `wsl --set-default-version 2`는 도커 엔진 구동에 필요한 WSL 버전 2를 기본값으로 고정하는 명령어입니다.

1. Windows 검색창에 `powershell` 입력 → **관리자로 실행** 클릭 → `wsl --install` 입력 후 완료되면 **PC 재부팅**

![Windows 검색창에서 PowerShell을 찾아 관리자 권한으로 실행합니다.](lms-img:0)

*Windows 검색창에서 PowerShell을 찾아 관리자 권한으로 실행합니다.*

![관리자 PowerShell에서 wsl --install 명령을 입력합니다.](lms-img:0)

*관리자 PowerShell에서 wsl --install 명령을 입력합니다.*

![WSL2와 Ubuntu 배포판이 함께 다운로드 및 설치됩니다.](lms-img:0)

*WSL2와 Ubuntu 배포판이 함께 다운로드 및 설치됩니다.*

2. 재부팅 후 열리는 Ubuntu 창에서 사용할 리눅스 계정 이름(`username`, 영문 소문자)과 비밀번호(`password`, 보안상 타이핑 시 화면에 글자가 표시되지 않으나 정상 입력됨)를 등록합니다.
3. PowerShell에서 `wsl --set-default-version 2`를 실행하고, 가상화 오류가 나면 **"Windows 기능 켜기/끄기"**에서 *Linux용 Windows 하위 시스템*과 *가상 머신 플랫폼* 두 항목을 체크한 뒤 재부팅합니다.

![재부팅 후 Ubuntu에서 사용할 리눅스 계정 이름(소문자)과 비밀번호를 설정합니다.](lms-img:0)

*재부팅 후 Ubuntu에서 사용할 리눅스 계정 이름(소문자)과 비밀번호를 설정합니다.*

### 2.2 [Windows WSL2 및 Ubuntu Linux 사용자] Docker Engine(`docker-ce`) 설치

**Windows 사용자는 PowerShell에서 `wsl`을 입력해 Ubuntu 리눅스 프롬프트(`계정@호스트:~$`)로 진입한 뒤, 아래 표의 5단계 명령어로 도커 엔진과 Compose 플러그인을 설치합니다.**

![PowerShell에서 wsl을 입력해 Ubuntu 리눅스 셸로 진입합니다.](lms-img:0)

*PowerShell에서 wsl을 입력해 Ubuntu 리눅스 셸로 진입합니다.*

<div style="font-size: 18px; line-height: 1.6;">

| 설치 단계 | 실행 명령어 (표에서 바로 복사 가능) | 명령어 및 옵션 설명 |
|---|---|---|
| **1. 저장소 준비** | `sudo apt-get update -y && sudo apt-get install -y ca-certificates curl` | 관리자 권한(`sudo`)으로 우분투 패키지 목록을 갱신하고, HTTPS로 도커 저장소에 접속하기 위한 인증서(`ca-certificates`)와 다운로드 도구(`curl`)를 설치합니다(`-y`: 자동 Yes 응답). |
| **2. 키링 폴더 생성 및 GPG 보안 키 등록** | `sudo install -m 0755 -d /etc/apt/keyrings`<br>`sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc`<br>`sudo chmod a+r /etc/apt/keyrings/docker.asc` | 권한 `0755`로 키링 폴더(`-d`)를 만들고, 도커 공식 파일임을 보증하는 암호화 서명 키(GPG)를 내려받아 읽기 권한(`a+r`)을 부여합니다. |
| **3. 도커 엔진 설치** | `sudo apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin` | 백그라운드 엔진(`docker-ce`), 터미널 CLI(`docker-ce-cli`), 런타임(`containerd.io`), 멀티 컨테이너 도구(`docker-compose-plugin`)를 한 번에 설치합니다. |
| **4. 사용자 권한 부여** | `sudo groupadd docker`<br>`sudo usermod -aG docker $USER` | 매번 `sudo`를 치지 않도록 현재 로그인한 계정(`$USER`)을 `docker` 그룹에 추가(`-aG`)합니다(터미널 재접속 또는 `newgrp docker` 입력 시 적용). |
| **5. 데몬 시작·확인** | `sudo service docker start`<br>`docker ps -a` | 백그라운드에서 도커 엔진 서비스를 깨우고, 컨테이너 목록 조회가 에러 없이 출력되는지 확인합니다. |

</div>

```bash
# 1~2단계: 필수 패키지 설치 및 Docker 공식 GPG 키·저장소 등록
sudo apt-get update -y
sudo apt-get install -y ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

echo   "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu   $(. /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}") stable" |   sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update -y

# 3~5단계: Docker Engine 설치, 권한 부여, 데몬 시작
sudo apt-get install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo groupadd docker 2>/dev/null || true
sudo usermod -aG docker $USER
sudo service docker start
```

![권한 설정 후 sudo 없이 docker ps가 정상 동작하는 화면입니다.](lms-img:0)

*권한 설정 후 sudo 없이 docker ps가 정상 동작하는 화면입니다.*

### 2.3 [macOS 사용자] Mac에서 도커 설치하기 (경량 CLI 방식 & Docker Desktop 방식)

**macOS 사용자는 기본 터미널이 유닉스(zsh/bash) 환경이므로 WSL 설치 없이 아래 두 방식 중 하나를 선택해 설치합니다.**

- **방식 A (Docker Desktop 없이 터미널 CLI만 쓰는 경량 방식 — `Homebrew` + `Colima`)**: 아래 코드에서 `brew install docker docker-compose colima`는 맥 패키지 관리자인 Homebrew로 도커 CLI와 경량 리눅스 런타임인 Colima를 설치하는 명령어이고, `colima start --cpu 2 --memory 4`는 CPU 2코어와 메모리 4GB를 할당해 도커 백그라운드 엔진을 시작하는 명령어입니다.
```bash
brew install docker docker-compose colima
colima start --cpu 2 --memory 4
docker --version && docker ps -a
```
- **방식 B (공식 `Docker Desktop for Mac` 설치)**: [Docker 공식 Mac 설치 페이지](https://docs.docker.com/desktop/setup/install/mac-install/)에서 본인 Mac 칩셋(Apple Silicon M1~M4 또는 Intel)에 맞는 `Docker.dmg`를 받아 `Applications` 폴더로 옮긴 뒤 실행(또는 `brew install --cask docker`)하고, 상단 메뉴바에 고래 아이콘이 뜨면 터미널에서 버전을 출력하는 `docker --version`을 실행해 확인합니다.

### 2.4 기본 실습 1: `hello-world`로 배우는 7대 필수 명령어 (`pull` · `images` · `run` · `ps` · `logs` · `rm` · `rmi`)

**`hello-world`는 도커가 정상 작동하는지 확인하기 위해 도커 공식 팀이 제공하는 초소형(약 13KB) 테스트용 이미지로, 실행 시 환영 메시지 한 문단을 출력하고 즉시 종료(`Exited (0)`)됩니다.** 실습에 앞서 이번 절에서 사용할 7대 필수 명령어와 옵션을 표로 먼저 살펴봅니다.

<div style="font-size: 18px; line-height: 1.6;">

| 명령어 및 주요 옵션 | 대상 | 역할과 동작 원리 |
|---|---|---|
| **`docker pull <이미지이름[:태그]>`** | 이미지 | 도커 허브(원격 레지스트리)에서 이미지를 내 PC로 내려받습니다. `:태그`를 생략하면 최신 버전(`:latest`)을 가져옵니다. |
| **`docker images`** | 이미지 | 내 PC(로컬)에 내려받아져 있거나 직접 빌드한 **모든 도커 이미지 목록**과 용량을 출력합니다. |
| **`docker run [--name 이름] <이미지>`** | 이미지 → 컨테이너 | 이미지를 바탕으로 **새 컨테이너를 생성하고 실행**합니다(`--name`을 생략하면 `stupefied_cori`, `elastic_johnson` 같은 `형용사_과학자이름` 조합이 랜덤 부여됨). |
| **`docker ps` / `docker ps -a`** | 컨테이너 | `docker ps`는 **현재 살아 있는(`Up`) 컨테이너**만 보여주고, **`-a`(`--all`)**를 붙이면 **실행이 끝나 꺼진(`Exited`) 컨테이너까지 전부** 보여줍니다. |
| **`docker logs <컨테이너이름/ID>`** | 컨테이너 | 컨테이너 내부 프로그램이 화면(표준 출력/에러)에 남긴 출력 기록을 확인합니다. |
| **`docker rm <컨테이너이름/ID>`** | **컨테이너 삭제** | 중지된 컨테이너를 삭제합니다(실행 중인 컨테이너를 강제로 지울 때는 `docker rm -f`). |
| **`docker rmi <이미지이름/ID>`** | **이미지 삭제** | 로컬 디스크에 저장된 이미지를 삭제합니다(**그 이미지를 참조하는 컨테이너가 단 1개라도 남아 있으면 `conflict` 에러로 삭제가 거부되므로 반드시 `docker rm`을 먼저 수행해야 합니다**). |

</div>

#### [Step 1] 이미지 내려받기(`pull`)와 실행(`run`), 목록 확인(`ps -a`)

```bash
# 1. 도커 허브에서 hello-world:latest 내려받기 및 로컬 이미지 목록 확인
docker pull hello-world
docker images | grep hello-world

# 2. 이름 없이 실행(랜덤 이름 자동 부여) vs --name으로 'test' 이름 지정 실행
docker run hello-world
docker run --name test hello-world

# 3. 종료된(Exited) 컨테이너까지 모두 목록 확인
docker ps -a
```

![docker ps -a로 확인한 컨테이너 목록입니다. 이름을 지정한 test와 자동 생성된 랜덤 이름(elastic_johnson)을 비교해 보십시오.](lms-img:0)

*docker ps -a로 확인한 컨테이너 목록입니다. 이름을 지정한 test와 자동 생성된 랜덤 이름(elastic_johnson)을 비교해 보십시오.*

#### [Step 2] 컨테이너 로그 확인(`logs`)과 올바른 삭제 순서(`rm` → `rmi`)

위에서 `docker run`을 두 번 실행했기 때문에, 현재 내 컴퓨터에는 **`test` 컨테이너 1개**와 **이름 없이 실행해 랜덤 이름이 붙은 컨테이너(예: `stupefied_cori`, `elastic_johnson`) 1개**, 총 2개의 컨테이너가 남아 있습니다.

만약 `docker rm test`로 `test`만 지우고 바로 `docker rmi hello-world`를 실행하면, **아직 남아 있는 랜덤 이름 컨테이너가 `hello-world` 이미지를 붙잡고 있어서 아래와 같은 `conflict` 에러가 발생합니다:**
> `Error response from daemon: conflict: unable to delete hello-world:latest (must be forced) - container ... is using its referenced image`

따라서 `docker ps -a`에서 확인한 랜덤 이름 컨테이너까지 모두 지우거나, 아래처럼 **`--filter ancestor=hello-world`로 해당 이미지에서 태어난 모든 컨테이너를 한 번에 찾아 지운 뒤** 이미지를 삭제합니다.

```bash
# 1. 종료된 test 컨테이너가 남긴 출력 로그 확인
docker logs test

# 2. test 컨테이너 삭제
docker rm test

# 3. hello-world 이미지로 만든 나머지 랜덤 이름 컨테이너까지 자동으로 모두 찾아 삭제
docker rm $(docker ps -aq --filter ancestor=hello-world)

# 4. 이제 참조하는 컨테이너가 0개이므로 hello-world 이미지가 깨끗하게 삭제됩니다!
docker rmi hello-world
```

![1.png](lms-img:0)

### 2.5 기본 실습 2: `nginx`로 배우는 백그라운드(`-dit`), 포트 연결(`-p`), 실시간 로그(`-f`), 내부 진입(`exec -it`)

**앞서 실행한 `hello-world`는 글자만 출력하고 0.1초 만에 꺼지지만, 우리가 만들 FastAPI 추론 서버나 웹 서버는 24시간 꺼지지 않고 떠서(`Up`) 손님의 요청을 기다려야 합니다.** 이번 실습에서는 전 세계에서 가장 널리 쓰이는 초경량 웹 서버인 **`nginx`(엔진엑스)** 를 띄워 서버 컨테이너를 다루는 법을 익힙니다.

<div style="font-size: 18px; line-height: 1.6;">

| 등장 도구 / 명령어 | 정체와 역할 | 왜 이번 실습과 LLM 서빙에서 필요한가 |
|---|---|---|
| **`nginx` (엔진엑스)** | 전 세계 웹사이트와 API 서버 앞단에서 가장 많이 쓰이는 **오픈소스 고성능 웹 서버 및 리버스 프록시**입니다(컨테이너 내부 기본 포트는 **`80`번**). | `hello-world`와 달리 **실행 후 죽지 않고 계속 살아 있는(`Up`) 서버 프로세스**의 가장 가벼운 표준 예제이며, 현업에서도 FastAPI/vLLM 추론 서버 앞단에서 트래픽을 받아 넘겨주는 문지기(51일차 게이트웨이 역할)로 널리 쓰입니다. |
| **`curl <주소>`** | 웹 브라우저 창을 띄우지 않고 **터미널 명령줄에서 바로 HTTP 요청을 보내 응답을 확인하는 도구**입니다. | 컨테이너 안쪽(`/bin/bash`)이나 그래픽 화면(GUI)이 없는 클라우드 리눅스 서버에서는 크롬 브라우저를 켤 수 없으므로, `curl localhost:80`처럼 서버가 잘 응답하는지 테스트할 때 필수로 사용합니다. |
| **`docker run -dit`** | `-d`(`--detach` 백그라운드) + `-i`(`--interactive` 표준입력 유지) + `-t`(`--tty` 가상 터미널 할당) | 죽지 않고 계속 도는 서버를 **백그라운드(`-d`)**로 띄워 내 터미널 입력창을 돌려받으면서, 나중에 안으로 접속할 수 있도록 입력 통로(`-it`)를 열어 둡니다. |
| **`-p 8000:80`** | `--publish <호스트포트>:<컨테이너내부포트>` | **내 PC(호스트)의 `8000`번 포트**로 들어온 요청을 **컨테이너 안쪽의 `80`번 포트(`nginx`)**로 이어 주는 다리(포트 포워딩)를 놓습니다. |
| **`docker logs -f <이름>`** | `--follow` (실시간 로그 스트리밍) | 로그를 한 번만 찍고 끝내지 않고, 새 접속 기록이 올라올 때마다 실시간으로 화면에 계속 출력합니다(**빠져나올 때는 `Ctrl+C`를 누르며, 로그 보기를 꺼도 컨테이너는 죽지 않습니다**). |
| **`docker stop <이름>`** | 실행 중인 컨테이너 안전 중지 | 백그라운드에서 돌고 있는(`Up`) 컨테이너에게 종료 신호를 보내 안전하게 멈춥니다(`Exited` 상태로 전환). |
| **`docker exec -it <이름> /bin/bash`** | 실행 중인 컨테이너 **내부로 직접 들어가** 배시(`bash`) 셸 실행 | 컨테이너 안쪽의 파일 목록을 확인하거나 컨테이너 내부 네트워크에서 직접 `curl` 테스트를 해 볼 때 사용합니다(**밖으로 빠져나올 때는 `exit` 입력**). |

</div>

#### [Step 1] 백그라운드(`-dit`)와 포트 연결(`-p 8000:80`)로 `nginx` 띄우고 접속하기

```bash
# 1. 컨테이너 내부 80번 포트를 내 PC(호스트) 8000번 포트에 연결해 백그라운드 실행
docker run -dit --name nginx -p 8000:80 nginx

# 2. STATUS가 Exited가 아니라 'Up ...' 상태이고 PORTS에 0.0.0.0:8000->80/tcp가 표시되는지 확인
docker ps -a
```

위 명령어를 실행한 뒤 내 PC의 웹 브라우저 주소창에 **[http://localhost:8000/](http://localhost:8000/)** 을 입력해 **"Welcome to nginx!"** 화면이 뜨는지 확인합니다.

![3.png](lms-img:0)

로컬호스트에서 nginx를 띄우는 데에 성공한 화면입니다

#### [Step 2] 실시간 접속 로그 추적하기 (`docker logs -f`)

```bash
# 실시간 로그 추적 시작 (실행한 채로 브라우저에서 http://localhost:8000 페이지를 새로고침(F5) 해 보세요!)
docker logs -f nginx
```

브라우저를 새로고침할 때마다 터미널에 `GET / HTTP/1.1 200` 로그가 실시간으로 올라오는 것을 확인한 뒤, 터미널에서 **`Ctrl + C`**를 눌러 로그 보기에서 빠져나옵니다.

![4.png](lms-img:0)

위의 형태로 로그를 확인할 수 있습니다

#### [Step 3] 기존 `nginx` 중지·삭제 후, 포트 연결(`-p`) 없이 띄워서 내부로 들어가 보기 (`exec -it`)

이번에는 **`-p 8000:80` 옵션을 빼고** 컨테이너를 띄우면 왜 내 PC 브라우저에서는 접속이 안 되는지, 그리고 컨테이너 안쪽(`exec -it`)으로 직접 들어가면 왜 접속이 되는지 비교해 봅니다.

```bash
# 1. 앞서 띄운 nginx 컨테이너를 중지하고 삭제합니다
docker stop nginx
docker rm -f nginx

# 2. 이번에는 -p 포트 연결 옵션 없이 nginx를 백그라운드로 실행합니다
docker run -dit --name nginx nginx
```

이 상태에서 내 PC 브라우저로 `http://localhost:8000/`에 접속하면 **사이트에 연결할 수 없음**이 뜹니다(컨테이너라는 밀폐된 방에 창문 `-p`를 안 뚫어 주었기 때문입니다). 이제 `docker exec -it`로 그 방 안으로 직접 들어가서 확인해 봅니다:

```bash
# 3. 실행 중인 nginx 컨테이너 내부의 bash 터미널로 진입합니다
docker exec -it nginx /bin/bash
```

프롬프트 모양이 `root@컨테이너ID:/#`로 바뀌면(컨테이너 안쪽 진입 성공!), 아래 두 줄을 차례로 입력합니다:

```bash
# 4. [컨테이너 내부 프롬프트에서 실행] 방 안쪽에서 자기 자신의 80번 포트를 호출하면 HTML 코드가 정상 출력됩니다!
curl localhost:80

# 5. 컨테이너 밖(내 원래 터미널)으로 빠져나옵니다
exit
```

```bash
# 6. 빠져나온 호스트 터미널에서 실습용 nginx 컨테이너와 이미지를 정리합니다
docker rm -f nginx
docker rmi nginx
```

![5.png](lms-img:0)

nginx의 웹페이지의 텍스트를 터미널에서 확인할 수 있습니다

### 체크포인트
1. `docker rmi`로 이미지를 지우기 전에 반드시 `docker rm`으로 컨테이너를 먼저 지워야 하는 이유는 무엇입니까? (목표 2)
2. `-p 8000:80` 없이 띄운 `nginx`가 호스트 브라우저에서는 안 열리지만 `docker exec -it nginx /bin/bash` 진입 후 `curl localhost:80`에는 응답하는 이유는 무엇입니까? (목표 1, 2)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. 종료(`Exited`)된 컨테이너라도 삭제되기 전까지는 해당 이미지 레이어를 참조하고 있어 이미지 삭제가 차단되기 때문입니다.
2. 컨테이너는 격리된 네트워크 공간을 가지므로 `-p`로 호스트 포트와 연결하지 않으면 밖에서 들어갈 수 없지만, `docker exec -it`는 컨테이너 내부 네트워크 안으로 직접 들어가 실행하므로 자기 자신의 `localhost:80`에 닿을 수 있습니다.

</details>

## 3. FastAPI 추론 서버 이미지 빌드와 레이어 캐시 최적화

## 3. FastAPI 추론 서버 이미지 빌드와 레이어 캐시 최적화 (`Dockerfile` & `dive`)

교재 : 6.1.2 2단계: LLM 컨테이너화하기

### 3.1 `Dockerfile` 7대 핵심 지시어와 `0.0.0.0` 바인딩 원리

**`Dockerfile`은 베이스 이미지(`FROM`) 위에 파일을 복사(`COPY`)하고 패키지를 설치(`RUN`)한 뒤 컨테이너 기동 명령(`CMD`)을 지정하는 이미지 빌드 설계도입니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 지시어 및 옵션 | 실행 시점 | 역할 및 핵심 규칙 |
|---|---|---|
| **`FROM python:3.11-slim`** | 빌드 시작 시 | 바탕이 될 베이스 이미지를 지정합니다. 기본 `python:3.11`(약 1GB) 대신 불필요한 문서·부가 도구를 덜어낸 경량 **`-slim` 버전(약 150MB)**을 쓰면 C/C++ 확장 패키지(`PyTorch`, `NumPy`) 호환성은 유지하면서 이미지 크기를 1/6로 줄일 수 있습니다. |
| **`WORKDIR </경로>`** | 빌드 시 | 컨테이너 내부의 기본 작업 폴더(예: `/app`)를 생성하고 이동합니다(이후 `COPY`, `RUN`, `CMD`는 모두 이 폴더 기준으로 실행됨). |
| **`COPY <호스트경로> <컨테이너경로>`** | 빌드 시 | 내 PC(호스트)의 파일·폴더를 이미지 안으로 복사합니다(`COPY requirements.txt .`의 끝 점 `.`은 컨테이너의 현재 `WORKDIR`인 `/app`을 뜻함). |
| **`RUN <명령어>`** | **빌드 시 1회** | 이미지 안에 패키지를 설치해 새 레이어로 저장합니다(`pip install --no-cache-dir`를 붙여 설치 후 남은 임시 다운로드 찌꺼기를 제거). |
| **`ENV <키>=<값>`** | 빌드 및 실행 시 | 컨테이너 내부 프로세스가 읽을 기본 환경 변수를 설정합니다(비밀 API 키는 절대 `ENV`에 적지 않음). |
| **`EXPOSE <포트>`** | 문서화 표시 | 컨테이너 내부 서버가 몇 번 포트를 쓰는지 명시합니다(실제 호스트 포트 연결은 `docker run -p`에서 수행). |
| **`CMD ["실행파일", "인자1", ...]`** | **컨테이너 기동 시** | 컨테이너가 켜질 때 실행할 메인 프로세스를 지정합니다. **반드시 `--host 0.0.0.0`으로 띄워야 컨테이너 외부에서 들어오는 요청을 받을 수 있습니다**(`127.0.0.1`로 띄우면 컨테이너 자기 자신만 접속 가능). |

</div>

46~49일차에서 배운 원칙(**기동 시 가중치 1회 로드 · `ThreadPoolExecutor` 비동기 분리 · `X-API-Key` 인증 · 로그 기록**)이 모두 담긴 프라이빗 추론 서버 코드를 생성합니다.

In [1]:
import json
from pathlib import Path

LAB = Path("docker_lab").resolve()
for sub in ["app", "frontend", "models", "logs"]:
    (LAB / sub).mkdir(parents=True, exist_ok=True)
(LAB / "app" / "__init__.py").write_text("", encoding="utf-8")

# 호스트의 models/ 폴더에 가중치 아티팩트 생성 (섹션 4에서 볼륨 마운트로 연결)
weights_data = {
    "model_name": "private-ko-sentiment-v1",
    "pos_weights": {"좋다": 1.2, "빠르다": 1.1, "안정": 1.3, "성공": 1.2, "편리": 1.0, "추천": 1.1},
    "neg_weights": {"오류": -1.3, "느리다": -1.1, "실패": -1.4, "중단": -1.2, "복잡": -0.9, "에러": -1.3},
    "bias": 0.1
}
(LAB / "models" / "sentiment_weights.json").write_text(
    json.dumps(weights_data, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("실습 폴더 및 가중치 준비 완료:", LAB)

실습 폴더 및 가중치 준비 완료: D:\vscode\aiffel\2026\11\02\docker_lab


In [2]:
%%writefile docker_lab/app/main.py
import os, json, time, asyncio, logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

MODEL_VERSION = os.environ.get("MODEL_VERSION", "1.0.0")
API_KEY = os.environ.get("SERVICE_API_KEY", "dev-secret-key")
MODEL_PATH = Path(os.environ.get("MODEL_PATH", "/app/models/sentiment_weights.json"))
LOG_DIR = Path(os.environ.get("LOG_DIR", "/app/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    handlers=[logging.FileHandler(LOG_DIR / "serving.log", encoding="utf-8"), logging.StreamHandler()]
)
logger = logging.getLogger("private_llm_api")

# GPU/CPU 추론 동시 실행 슬롯을 통제하는 전용 스레드풀 (D48 핵심 패턴)
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="infer")

def load_weights():
    if MODEL_PATH.exists():
        return json.loads(MODEL_PATH.read_text(encoding="utf-8"))
    # 단일 컨테이너 기초 실습(볼륨 미마운트 시)을 위한 기본 가중치 폴백
    return {"model_name": "builtin-fallback", "pos_weights": {"좋다": 1.0, "안정": 1.0}, "neg_weights": {"오류": -1.0, "에러": -1.0}, "bias": 0.0}

MODEL_ARTIFACT = load_weights()

app = FastAPI(title="Private Model Serving API", version=MODEL_VERSION)

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=1000, description="분석할 한국어 문장")

def run_sync_inference(text: str) -> dict:
    score = MODEL_ARTIFACT.get("bias", 0.0)
    for w, weight in MODEL_ARTIFACT["pos_weights"].items():
        if w in text:
            score += weight
    for w, weight in MODEL_ARTIFACT["neg_weights"].items():
        if w in text:
            score += weight
    label = "positive" if score >= 0 else "negative"
    return {"label": label, "score": round(score, 3), "model_source": MODEL_ARTIFACT["model_name"]}

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_version": MODEL_VERSION,
        "model_source": MODEL_ARTIFACT["model_name"],
        "weights_mounted": MODEL_PATH.exists()
    }

@app.post("/predict")
async def predict(req: PredictRequest, x_api_key: str = Header(default="")):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="유효하지 않은 API Key입니다.")
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(inference_executor, run_sync_inference, req.text)
    logger.info(f"predict_ok label={result['label']} score={result['score']} source={result['model_source']}")
    return {**result, "model_version": MODEL_VERSION}

Writing docker_lab/app/main.py


In [3]:
%%writefile docker_lab/requirements.txt
fastapi==0.115.0
uvicorn[standard]==0.30.6
pydantic==2.9.2
requests==2.32.3

Writing docker_lab/requirements.txt


### 3.2 레이어 캐시(Layer Cache) 원리와 `Dockerfile.bad` vs `Dockerfile` 비교 실험

**도커는 바뀐 레이어 아래쪽의 모든 레이어 캐시를 무효화하므로, 거의 바뀌지 않는 `COPY requirements.txt`와 `RUN pip install`을 먼저 두고 수시로 고치는 `COPY app/`을 맨 뒤에 두어야 코드 수정 시 패키지 재설치 없이 1초 만에 재빌드됩니다.**

![바뀐 레이어 위로는 전부 새로 만들므로 자주 바뀌는 것을 위에 쌓아야 비싼 설치 레이어가 캐시에 남습니다.](lms-img:0)

*바뀐 레이어 위로는 전부 새로 만들므로 자주 바뀌는 것을 위에 쌓아야 비싼 설치 레이어가 캐시에 남습니다.*

레이어 순서가 실제 재빌드 속도와 이미지 구조에 미치는 차이를 직접 실험하기 위해, **나쁜 설계(`Dockerfile.bad`)**와 **최적화 설계(`Dockerfile`)** 두 파일을 모두 생성합니다.

In [4]:
%%writefile docker_lab/Dockerfile.bad
FROM python:3.11-slim
WORKDIR /app

# [나쁜 예] 소스 코드 전체를 먼저 복사하면 코드 한 줄만 고쳐도 아래 pip install 캐시가 매번 깨집니다!
COPY app/ ./app/
COPY requirements.txt .
RUN pip install -r requirements.txt

ENV MODEL_VERSION="1.0.0"
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing docker_lab/Dockerfile.bad


In [5]:
%%writefile docker_lab/Dockerfile
FROM python:3.11-slim
WORKDIR /app

# [권장 표준] 1. 패키지 목록만 먼저 복사해 무거운 pip install 레이어를 캐싱하고 --no-cache-dir로 용량을 줄입니다
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 2. 자주 바뀌는 애플리케이션 코드는 맨 뒤에 복사합니다
COPY app/ ./app/

ENV MODEL_VERSION="1.0.0"
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing docker_lab/Dockerfile


### 3.3 [터미널 실습] `bad` vs `good` 재빌드 속도 비교, `dive` 레이어 분석, 컨테이너 운영 명령어

**작성한 두 도커파일을 직접 빌드해 캐시 속도 차이를 비교하고, 이미지 레이어 분석 도구인 `dive`와 운영 명령어(`tag`, `start`, `stats`, `system prune`)를 실습합니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 명령어 및 도구 | 구성 요소 해부 | 역할과 동작 원리 |
|---|---|---|
| **`docker build . -f <파일명> -t <이름:태그>`** | `docker build` + **`.`(현재 폴더)** + `-f` + `-t` | 현재 폴더(`.`)의 파일들을 묶어 이미지를 빌드합니다. **점(`.`)을 `docker build .` 바로 뒤에 적으면 마우스로 복사할 때 맨 끝의 점이 누락되는 실수를 막을 수 있습니다.** |
| **`time <명령어>`** | 리눅스 기본 시간 측정 명령어 | 뒤에 적은 명령어가 시작부터 끝날 때까지 실제로 총 몇 초(`real`)가 걸렸는지 소수점 단위로 측정해 출력합니다. |
| **`dive <이미지이름>`** | 오픈소스 도커 이미지 레이어 분석기 | 이미지를 이루는 각 층(레이어)마다 어떤 파일이 추가·삭제되었는지 트리 구조로 보여 주고, 불필요한 중복 파일로 낭비된 용량이 없는지 **효율성 점수(`Efficiency Score %`)**를 매겨 줍니다(종료: `Ctrl+C`). |
| **`docker tag <원본> <새이름:새태그>`** | 이미지 별칭(태그) 부여 | 이미지를 새로 복제하지 않고 동일한 이미지 ID에 배포용 새 이름·버전 태그를 붙입니다. |
| **`docker start <컨테이너이름>`** | 중지된 컨테이너 재가동 | 꺼진(`Exited`) 기존 컨테이너를 새로 만들지 않고 백그라운드 상태로 다시 켭니다. |
| **`docker stats --no-stream`** | 실시간 자원 모니터링 | 실행 중인 컨테이너의 **CPU·메모리(RAM) 사용량과 한도**를 표로 출력합니다(`--no-stream`: 1회 즉시 출력). |
| **`docker system prune -a`** | 도커 디스크 일괄 청소 | 중지된 모든 컨테이너, 안 쓰는 네트워크, 빌드 캐시, 어떤 컨테이너도 참조하지 않는 이미지를 한 번에 지워 디스크 용량을 확보합니다. |

</div>

#### [Step 1] `Dockerfile.bad`와 `Dockerfile` 최초 빌드 후 코드 수정 시 재빌드 속도 비교

```bash
cd docker_lab

# 1. 나쁜 설계(bad)와 좋은 설계(good) 최초 빌드 (점 '.'이 build 바로 뒤에 있습니다)
docker build . -f Dockerfile.bad -t fastapi-app:bad
docker build . -f Dockerfile -t fastapi-app:good

# 2. 두 이미지의 용량 비교 (--no-cache-dir 유무에 따른 크기 차이 확인)
docker images | grep fastapi-app
```

이제 `app/main.py` 파일 맨 끝에 주석 한 줄을 추가한 뒤, **`time` 명령어로 두 도커파일의 재빌드 소요 시간을 비교**해 봅니다:

```bash
# 3. 소스 코드 한 줄 수정
echo "# code modified" >> app/main.py

# 4. [나쁜 설계 재빌드] COPY app/이 맨 위에 있어 pip install 캐시가 깨지고 처음부터 다시 설치합니다 (수십 초 소요)
time docker build . -f Dockerfile.bad -t fastapi-app:bad

# 5. [좋은 설계 재빌드] pip install 레이어가 CACHED 처리되어 1초 이내에 즉시 끝납니다!
time docker build . -f Dockerfile -t fastapi-app:good
```

![6.png](lms-img:0)

Bad file (위) 은 10초 정도 걸리는 반면, Good file (아래) 은 즉각 빌드가 완료되었습니다

![7.png](lms-img:0)

#### [Step 2] 컨테이너 실행, 리소스 사용량(`docker stats`) 확인 및 `dive` 레이어 검사

```bash
# 1. 최적화한 fastapi-app:good 이미지로 컨테이너를 백그라운드 실행합니다
docker run -dit --name fastapi-app -p 127.0.0.1:8000:8000 fastapi-app:good

# 2. 컨테이너가 사용하는 메모리(MEM USAGE)와 CPU 비율을 확인합니다
docker stats --no-stream fastapi-app

# 3. 헬스체크 API가 정상 응답하는지 확인합니다
curl http://127.0.0.1:8000/health
```

```bash
# 4. [선택 실습] dive 도구 설치 및 이미지 레이어 효율성 점수(Efficiency Score) 확인
# - macOS: brew install dive
# - Ubuntu / WSL2:
DIVE_VERSION=$(curl -sL "https://api.github.com/repos/wagoodman/dive/releases/latest" | grep '"tag_name":' | sed -E 's/.*"v([^"]+)".*//')
curl -fOL "https://github.com/wagoodman/dive/releases/download/v${DIVE_VERSION}/dive_${DIVE_VERSION}_linux_amd64.deb"
sudo apt-get install -y ./dive_${DIVE_VERSION}_linux_amd64.deb

# dive 실행 (화살표 키로 레이어 이동, 종료는 Ctrl + C)
dive fastapi-app:good
```

### 체크포인트
1. `Dockerfile`의 `RUN`과 `CMD`가 실행되는 시점 차이를 설명하십시오. (목표 3)
2. `Dockerfile.bad`처럼 `COPY app/ ./app/`을 `RUN pip install` 앞에 두면 코드 한 줄만 고쳐도 빌드가 느려지는 이유는 무엇입니까? (목표 3)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. **`RUN`**은 `docker build`로 이미지를 만들 때 1회 실행되어 이미지 레이어로 굳어지고, **`CMD`**는 `docker run`/`start`로 컨테이너가 켜질 때마다 실행됩니다.
2. 코드가 바뀌어 앞선 `COPY app/` 레이어의 해시가 달라지면, 도커는 그 아래에 있는 `RUN pip install` 레이어 캐시까지 모두 무효화하고 패키지를 처음부터 다시 설치하기 때문입니다.

</details>

## 4. 프라이빗 LLM을 위한 시크릿 분리, 가중치 볼륨 마운트, GPU 연결

## 4. 프라이빗 LLM을 위한 시크릿 분리(`.dockerignore`), 가중치 볼륨 마운트(`-v`), GPU 연결(`--gpus`)

교재 : 6.1.2 LLM 컨테이너화하기, 6.4 자격 증명 관리

### 4.1 빌드 컨텍스트(Build Context)와 `.dockerignore`, 시크릿 잔류 원리 (`RUN rm`의 함정)

**`docker build .`을 실행하면 도커는 `Dockerfile`을 읽기도 전에 현재 폴더(`.`) 안의 모든 파일과 하위 폴더를 통째로 묶어 도커 엔진으로 전송하는데, 이 전송 묶음을 '빌드 컨텍스트(Build Context)'라고 부릅니다.**

따라서 `.dockerignore` 파일이 없으면 두 가지 심각한 문제가 발생합니다:
1. **빌드 속도 마비**: 설령 `Dockerfile`에 `COPY models/`를 적지 않았더라도, 현재 폴더 안에 10GB짜리 `models/`나 `.venv/` 폴더가 들어 있으면 빌드할 때마다 10GB를 도커 데몬으로 복사하느라 시작부터 수 분씩 멈춥니다.
2. **시크릿 영구 잔류**: 앞 레이어에서 `COPY . .`로 들어간 비밀 파일(`.env`, `api_keys.json`)을 뒷 레이어에서 `RUN rm`으로 지워도 **아래쪽 읽기 전용 레이어에는 원본 파일이 그대로 남아 추출**될 수 있고, `ENV`/`ARG`에 적은 키는 `docker history --no-trunc <이미지>` 명령에 평문으로 노출됩니다.

아래 `.dockerignore`에서 **`**/`**는 '모든 하위 폴더 깊이'를 뜻하므로, `**/.env`라고 적으면 최상위 `.env`뿐 아니라 `app/.env`처럼 안쪽 폴더에 들어 있는 비밀 파일까지 빌드 컨텍스트 전송 단계에서 원천 제외됩니다.

![지운 파일은 새 레이어에 삭제 표시만 남기고 아래 레이어에 그대로 있으며 ENV와 ARG로 넣은 키는 이미지 설정과 빌드 기록에 남습니다.](lms-img:0)

*지운 파일은 새 레이어에 삭제 표시만 남기고 아래 레이어에 그대로 있으며 ENV와 ARG로 넣은 키는 이미지 설정과 빌드 기록에 남습니다.*

In [6]:
%%writefile docker_lab/.dockerignore
.git
__pycache__/
.venv/
**/.env
**/api_keys.json
models/
logs/

Writing docker_lab/.dockerignore


### 4.2 프라이빗 LLM 서빙의 핵심: 대용량 가중치 볼륨 마운트(`-v`)와 GPU 패스스루(`--gpus all`)

**수 GB~수십 GB에 달하는 프라이빗 LLM 가중치(`models/`)를 `Dockerfile`에서 `COPY`로 이미지 안에 넣으면 이미지 하나가 15GB 이상으로 비대해지고 모델을 바꿀 때마다 전체 이미지를 다시 빌드해야 하므로, 가중치는 `.dockerignore`로 이미지에서 제외한 뒤 컨테이너 실행 시점에 호스트 디스크를 볼륨(`-v`)으로 마운트합니다.**

![대용량 모델 가중치는 이미지에 COPY하지 않고 호스트 볼륨(-v :ro)으로 연결하며, 물리 GPU는 --gpus all로 패스스루합니다.](lms-img:0)

*대용량 모델 가중치는 이미지에 COPY하지 않고 호스트 볼륨(-v :ro)으로 연결하며, 물리 GPU는 --gpus all로 패스스루합니다.*

프라이빗 LLM 엔지니어가 컨테이너에 모델 가중치, 설정 파일, 물리 GPU를 연결할 때 사용하는 핵심 옵션은 다음과 같습니다.

<div style="font-size: 18px; line-height: 1.6;">

| 옵션 및 설정 | 문법 구조 | 역할과 프라이빗 LLM 실무 기준 |
|---|---|---|
| **`-v $(pwd)/models:/app/models:ro`** | `--volume <호스트절대경로>:<컨테이너경로>:ro` | 호스트의 모델 가중치 폴더를 컨테이너 내부 `/app/models`에 연결합니다. 도커 `-v`는 호스트 경로에 **절대 경로**를 요구하므로 현재 작업 폴더의 전체 경로로 자동 치환되는 **`$(pwd)`**를 앞에 붙이며, 끝에 **`:ro`(읽기 전용)**를 붙여 컨테이너가 원본 가중치를 훼손하지 못하게 잠급니다. |
| **`-v $(pwd)/logs:/app/logs`** | 읽기·쓰기 볼륨 마운트 | 컨테이너가 실행 중 남기는 추론 로그를 호스트의 `./logs` 폴더에 실시간 저장해 컨테이너 삭제(`docker rm`) 후에도 로그가 영구 보존되게 합니다. |
| **`--env-file .env`** | 환경 변수 파일 주입 | 이미지 안에 하드코딩하지 않은 비밀 키(`SERVICE_API_KEY`)와 모델 버전 설정을 호스트의 `.env` 파일에서 읽어 컨테이너 메모리에만 안전하게 주입합니다. |
| **`--gpus all`** | `docker run --gpus all` | 호스트에 **NVIDIA Container Toolkit(`nvidia-ctk`)**이 설치된 환경에서 호스트의 물리 GPU를 컨테이너 안으로 직접 패스스루(연결)합니다(53일차 로컬 GPU 배포 및 57일차 `vLLM` 서빙에서 사용). |

</div>

호스트의 `.env` 파일을 생성한 뒤, 기존 컨테이너를 지우고 **가중치 볼륨(`:ro`), 로그 볼륨, `.env` 파일**을 연결해 컨테이너를 다시 띄워 봅니다.

In [7]:
%%writefile docker_lab/.env
MODEL_VERSION=1.1.0-private
SERVICE_API_KEY=modu-private-llm-key

Writing docker_lab/.env


```bash
# 1. 기존 단일 컨테이너 삭제
docker rm -f fastapi-app 2>/dev/null || true

# 2. 가중치 폴더(:ro 읽기 전용) + 로그 폴더 + .env를 마운트해 컨테이너 실행
docker run -dit --name fastapi-app   -p 127.0.0.1:8000:8000   --env-file .env   -v $(pwd)/models:/app/models:ro   -v $(pwd)/logs:/app/logs   fastapi-app:good

# 3. 헬스체크: weights_mounted가 true이고 model_source가 "private-ko-sentiment-v1"로 바뀌었는지 확인!
curl http://127.0.0.1:8000/health
```

```bash
# 4. 추론 요청 후 호스트의 ./logs/serving.log 파일에 영구 로그가 남았는지 확인!
curl -X POST http://127.0.0.1:8000/predict   -H "Content-Type: application/json"   -H "X-API-Key: modu-private-llm-key"   -d '{"text": "프라이빗 컨테이너 배포가 빠르다 그리고 안정적이다"}'

cat logs/serving.log
```

### 체크포인트
1. `.dockerignore`에 `.env` 대신 `**/.env`로 적어야 하는 이유와, `RUN rm -f .env`로 지워도 키가 유출되는 이유를 설명하십시오. (목표 3)
2. 프라이빗 LLM 서빙에서 `models/` 폴더를 이미지에 `COPY`하지 않고 `-v $(pwd)/models:/app/models:ro`로 마운트하는 이유 두 가지를 설명하십시오. (목표 1, 3)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. `**/.env`로 적어야 하위 폴더(`app/.env` 등)의 파일까지 모두 제외됩니다. 또한 이미지 레이어는 불변이므로 `RUN rm`은 맨 위 레이어에서 파일을 가릴 뿐 아래쪽 `COPY` 레이어에 원본이 그대로 남아 유출됩니다.
2. 첫째, 수십 GB짜리 가중치가 이미지에 들어가 이미지 크기가 비대해지고 모델 변경 시마다 재빌드해야 하는 문제를 막기 위함이고, 둘째, `:ro`(읽기 전용) 옵션으로 컨테이너 안에서 원본 가중치 파일이 변조·삭제되는 사고를 방지하기 위함입니다.

</details>

## 5. 포트, 볼륨, Docker Compose로 다중 컨테이너 조립하기

교재 : 6.7.2 마이크로서비스 기반 아키텍처 구현, 6.11 다중모델 오케스트레이션

### 5.1 Docker Compose란 무엇이며, 왜 컨테이너를 2개로 나누어 묶는가?

**49일차에서 우리는 화면을 그리는 `Streamlit` 서버(`8501`번)와 모델을 돌리는 `FastAPI` 서버(`8000`번)를 분리했습니다.** 이때 최종 사용자(손님)는 웹 브라우저만 켜서 접속할 뿐 파이썬이나 도커를 설치하지 않으며, 우리 서버 컴퓨터 안에서 **가벼운 화면용 컨테이너(`ui`, PyTorch 미설치)**와 **무거운 추론용 컨테이너(`api`, 모델·런타임 포함)** 두 개가 함께 돌아갑니다.

**Docker Compose(도커 컴포즈)는 이처럼 여러 개의 컨테이너(`api`, `ui`, `db` 등)와 네트워크·볼륨 설정을 `compose.yaml` 파일 한 장에 정의해 두고, 명령어 한 줄(`docker compose up`)로 동시에 켜고 끄는 다중 컨테이너 오케스트레이션 도구입니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 구분 | 역할과 비유 | 다루는 범위와 실행 명령어 |
|---|---|---|
| **`Dockerfile`** | **컨테이너 1개짜리 제조 레시피**<br>(어느 베이스 이미지에 어떤 패키지를 깔지 정의) | 단일 이미지 1개 생성 (`docker build . -t ...`) |
| **`compose.yaml`** | **여러 컨테이너의 전체 상차림·배치도**<br>(`api`와 `ui`를 어떤 포트·볼륨·네트워크로 엮어 함께 띄울지 정의) | 여러 컨테이너 + 가상 네트워크 일괄 기동·종료 (`docker compose up` / `down`) |

</div>

만약 `Docker Compose` 없이 `docker run`만 두 번 쳐서 `api`와 `ui`를 띄우려면 ① 매번 5~6줄짜리 긴 옵션을 컨테이너마다 따로 입력해야 하고, ② 두 컨테이너가 대화할 가상 네트워크(`docker network create`)를 수동으로 만들고 연결해야 하며, ③ 기동 순서(`depends_on`)를 맞출 수 없습니다. `compose.yaml`을 쓰면 이 모든 과정이 자동화됩니다.

### 5.2 호스트 `127.0.0.1` 바인딩과 Compose 내부 서비스 이름(`http://api:8000`)

**다중 컨테이너(`api` + `ui`)를 `compose.yaml`로 묶을 때는 세 가지 네트워크 주소를 '누구 입장에서 부르는가'로 구분해야 합니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 설정 위치 | 올바른 값 | 잘못 설정했을 때 발생하는 현상 |
|---|---|---|
| **1. 컨테이너 내부 서버 기동** (`Dockerfile` `CMD`) | **`0.0.0.0`** | `127.0.0.1`로 띄우면 컨테이너 자기 방 안에서만 듣고 외부 연결을 모두 거부함 |
| **2. 호스트 포트 바인딩** (`compose.yaml` `ports`) | **`"127.0.0.1:8000:8000"`** | `"8000:8000"`으로 열면 도커가 `iptables`를 조작해 **`ufw` 방화벽을 우회하고 외부 망에 포트를 개방함** |
| **3. 컨테이너 간 내부 통신** (`ui` → `api` 호출) | **`http://api:8000`** | `ui`에 `http://127.0.0.1:8000`을 넣으면 `ui` 컨테이너 자기 안을 뒤지다 `ConnectionRefusedError` 발생 |

</div>

![주소를 붙이지 않고 연 포트(0.0.0.0:8000)는 리눅스 ufw 방화벽을 우회해 외부 망의 모든 기기에 노출됩니다.](lms-img:0)

*주소를 붙이지 않고 연 포트(0.0.0.0:8000)는 리눅스 ufw 방화벽을 우회해 외부 망의 모든 기기에 노출됩니다.*

![화면(ui)은 추론 서버를 서비스 이름(http://api:8000)으로 호출하고, 설정은 .env에서, 가중치와 로그는 볼륨 마운트로 연결합니다.](lms-img:0)

*화면(ui)은 추론 서버를 서비스 이름(http://api:8000)으로 호출하고, 설정은 .env에서, 가중치와 로그는 볼륨 마운트로 연결합니다.*

### 5.3 `compose.yaml` 핵심 지시어와 `docker compose` 세부 명령어·옵션

**`compose.yaml` 파일 안에 적는 핵심 지시어와 터미널에서 실행하는 `docker compose` 명령어 조각들의 의미는 다음과 같습니다.**

<div style="font-size: 18px; line-height: 1.6;">

| 구분 | 지시어 / 명령어 조각 | 역할과 동작 원리 |
|---|---|---|
| **`compose.yaml` 지시어** | **`services:`** | 함께 띄울 컨테이너 단위(예: `api`, `ui`)들을 정의하는 최상위 블록입니다. 여기서 적은 **서비스 이름(`api`)이 곧 컨테이너 간 통신용 내부 도메인 주소(`http://api:8000`)**가 됩니다. |
| **`compose.yaml` 지시어** | **`build: {context: ., dockerfile: ...}`** | 이미지를 빌드할 기준 폴더(`context`)와 사용할 도커파일 이름(`Dockerfile`, `Dockerfile.ui`)을 지정합니다. |
| **`compose.yaml` 지시어** | **`ports: ["127.0.0.1:8000:8000"]`** | 호스트의 로컬 루프백(`127.0.0.1`) 포트를 컨테이너 내부 포트와 안전하게 연결합니다. |
| **`compose.yaml` 지시어** | **`env_file: [".env"]` & `environment:`** | `.env` 파일의 비밀 키·설정을 일괄 주입(`env_file`)하거나 개별 환경 변수(`environment`)를 컨테이너에 전달합니다. |
| **`compose.yaml` 지시어** | **`volumes: ["./models:/app/models:ro"]`** | 호스트의 `./models` 폴더를 읽기 전용(`:ro`)으로, `./logs` 폴더를 읽기·쓰기용으로 마운트합니다. |
| **`compose.yaml` 지시어** | **`depends_on: [api]`** | `ui` 컨테이너를 띄우기 전에 `api` 컨테이너를 먼저 기동하도록 시작 순서를 지정합니다. |
| **`docker compose` 명령** | **`docker compose`** | 현재 폴더의 `compose.yaml` 파일을 읽어 다중 컨테이너를 제어하는 도커 공식 플러그인을 호출합니다(다른 이름의 파일을 쓸 때는 `-f 파일명` 추가). |
| **`up` 서브커맨드** | **`up`** + **`-d`** + **`--build`** | `compose.yaml`의 모든 컨테이너와 전용 가상 네트워크를 생성·시작(`up`)하되, 시작 전 이미지를 새로 빌드(`--build`)하고 백그라운드(`-d`)에서 실행합니다. |
| **`ps` / `logs` 서브커맨드** | **`docker compose ps`** / **`logs -f`** | 현재 Compose 프로젝트에 속한 컨테이너들의 실행 상태(`ps`)와 실시간 통합 로그(`logs -f`)를 출력합니다. |
| **`down` 서브커맨드** | **`docker compose down --volumes`** | 실행 중인 모든 컨테이너를 중지·삭제하고 전용 가상 네트워크와 임시 볼륨(`--volumes`)까지 한 번에 철거합니다. |

</div>

In [8]:
%%writefile docker_lab/frontend/app.py
import os, requests, streamlit as st

API_BASE = os.environ.get("API_BASE_URL", "http://api:8000")
st.set_page_config(page_title="프라이빗 모델 서빙 대시보드", layout="wide")
st.title("프라이빗 모델 서빙 대시보드 (Docker Compose)")

api_key = st.sidebar.text_input("X-API-Key", type="password", value="modu-private-llm-key")
if st.sidebar.button("서버 헬스체크 (/health)"):
    try:
        hr = requests.get(f"{API_BASE}/health", timeout=3)
        st.sidebar.json(hr.json())
    except Exception as e:
        st.sidebar.error(f"연결 실패: {e}")

text = st.text_area("분석할 문장 입력", value="컨테이너 볼륨 마운트로 가중치를 안전하고 빠르게 서빙합니다.")
if st.button("추론 요청 (/predict)"):
    try:
        r = requests.post(
            f"{API_BASE}/predict",
            json={"text": text},
            headers={"X-API-Key": api_key},
            timeout=5
        )
        st.write(f"HTTP 상태 코드: `{r.status_code}`")
        st.json(r.json())
    except Exception as e:
        st.error(f"API 호출 오류: {e}")

Writing docker_lab/frontend/app.py


In [9]:
%%writefile docker_lab/Dockerfile.ui
FROM python:3.11-slim
WORKDIR /app
RUN pip install --no-cache-dir streamlit==1.39.0 requests==2.32.3
COPY frontend/ ./frontend/
EXPOSE 8501
CMD ["streamlit", "run", "frontend/app.py", "--server.address=0.0.0.0", "--server.port=8501"]

Writing docker_lab/Dockerfile.ui


In [10]:
%%writefile docker_lab/compose.yaml
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "127.0.0.1:8000:8000"
    env_file:
      - .env
    volumes:
      - ./models:/app/models:ro
      - ./logs:/app/logs
    # [참고] NVIDIA GPU가 장착된 호스트에서 GPU를 컨테이너에 연결할 때 아래 주석을 해제합니다 (D53, D57 연계)
    # deploy:
    #   resources:
    #     reservations:
    #       devices:
    #         - driver: nvidia
    #           count: all
    #           capabilities: [gpu]

  ui:
    build:
      context: .
      dockerfile: Dockerfile.ui
    ports:
      - "127.0.0.1:8501:8501"
    environment:
      - API_BASE_URL=http://api:8000
    depends_on:
      - api

Writing docker_lab/compose.yaml


#### [Step 1] `docker compose up -d --build`로 `api` + `ui` 동시 기동 및 확인

```bash
# 1. 앞서 단독으로 띄웠던 단일 컨테이너가 있다면 포트가 겹치지 않게 삭제합니다
docker rm -f fastapi-app 2>/dev/null || true

# 2. compose.yaml을 읽어 api와 ui 이미지를 빌드하고 백그라운드로 동시 기동합니다
docker compose up -d --build

# 3. api(8000번)와 ui(8501번) 두 컨테이너가 모두 Up 상태인지 확인합니다
docker compose ps
```

이제 웹 브라우저에서 **[http://localhost:8501](http://localhost:8501)** 에 접속해 사이드바의 **[서버 헬스체크]** 버튼과 본문의 **[추론 요청]** 버튼을 눌러 `ui` 컨테이너가 `http://api:8000`으로 백엔드를 호출해 결과를 받아오는지 확인합니다.

![9.png](lms-img:0)

![8.png](lms-img:0)

화면과, 실행로그입니다

#### [Step 2] 실습 확인 후 일괄 종료 및 네트워크·볼륨 정리

```bash
# 실행 중인 api, ui 컨테이너와 전용 가상 네트워크를 한 번에 종료·삭제합니다
docker compose down --volumes
```

### 체크포인트
1. 호스트 포트 바인딩을 `"8000:8000"` 대신 `"127.0.0.1:8000:8000"`으로 지정해야 하는 보안상 이유와, `ui` 컨테이너가 `api`를 부를 때 `http://api:8000`을 쓰는 이유를 설명하십시오. (목표 3)

<details class="lms-answer">
<summary>정답 및 해설 확인</summary>

1. `"8000:8000"`은 `0.0.0.0`에 바인딩되어 리눅스 `ufw` 방화벽을 우회해 외부 망에 노출되므로 `"127.0.0.1:8000:8000"`으로 막아야 합니다. 또한 `ui` 컨테이너 안에서 `127.0.0.1`은 `ui` 자신이므로 Compose 내부 DNS 이름인 `http://api:8000`을 호출해야 합니다.

</details>

## 6. [오후 실습 미션] 고장 난 컨테이너 3대 장애 해결과 도커라이징 검증

### 6.1 현업 도커 서빙에서 가장 자주 발생하는 3대 설정 장애

**컨테이너가 정상 실행(`Up`) 중인데도 접속이 안 되거나 `503`·`ConnectionError`가 발생하는 원인의 90%는 바인딩 주소(`0.0.0.0`), 내부 DNS 주소(`http://api:8000`), 볼륨 마운트 경로 불일치 세 가지입니다.**

![도커 서빙에서 가장 자주 겪는 3대 장애(컨테이너 내부 127.0.0.1 바인딩, UI의 localhost 호출, 모델 볼륨 누락)와 해결 지점입니다.](lms-img:0)

*도커 서빙에서 가장 자주 겪는 3대 장애(컨테이너 내부 127.0.0.1 바인딩, UI의 localhost 호출, 모델 볼륨 누락)와 해결 지점입니다.*

아래 코드 셀을 실행하면 세 가지 설정 버그가 심어진 `docker_lab/compose.broken.yaml`과 `docker_lab/Dockerfile.broken`이 생성됩니다.

<div style="font-size: 18px; line-height: 1.6;">

| 미션 번호 | 증상 (실행 시 나타나는 에러) | 진단 명령어 | 찾아서 고쳐야 할 파일 및 항목 |
|---|---|---|---|
| **장애 미션 1** | 컨테이너는 `Up`인데 호스트에서 `curl http://127.0.0.1:8000/health` 호출 시 **`Recv failure: Connection reset by peer` / `Empty reply from server`** 발생 | `docker compose -f compose.broken.yaml logs api` | `Dockerfile.broken`의 `CMD`에서 `--host` 바인딩 주소 수정 |
| **장애 미션 2** | Streamlit 화면(`http://localhost:8501`)에서 버튼 클릭 시 **`ConnectionError: HTTPConnectionPool(host='localhost', port=8000)`** 발생 | `docker compose -f compose.broken.yaml logs ui` | `compose.broken.yaml`의 `ui` 서비스 `API_BASE_URL` 환경 변수 수정 |
| **장애 미션 3** | `/health` 응답에서 **`"weights_mounted": false`**가 나오고 가중치 파일(`sentiment_weights.json`)을 읽지 못함 | `docker exec -it <api컨테이너> ls -la /app/models` | `compose.broken.yaml`의 `api` 서비스 `volumes` 마운트 경로 수정 |

</div>

In [11]:
%%writefile docker_lab/Dockerfile.broken
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
EXPOSE 8000
# [버그 1] 컨테이너 내부에서 127.0.0.1로 띄우면 컨테이너 밖에서 들어오는 패킷을 모두 거부합니다!
CMD ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"]

Writing docker_lab/Dockerfile.broken


In [12]:
%%writefile docker_lab/compose.broken.yaml
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile.broken
    ports:
      - "127.0.0.1:8000:8000"
    env_file:
      - .env
    volumes:
      # [버그 3] 컨테이너 내부 코드는 /app/models/sentiment_weights.json을 찾는데 잘못된 경로(/app/wrong_models)에 마운트됨!
      - ./models:/app/wrong_models:ro
      - ./logs:/app/logs

  ui:
    build:
      context: .
      dockerfile: Dockerfile.ui
    ports:
      - "127.0.0.1:8501:8501"
    environment:
      # [버그 2] ui 컨테이너 안에서 localhost:8000을 부르면 ui 컨테이너 자기 자신을 찌르다가 실패합니다!
      - API_BASE_URL=http://localhost:8000
    depends_on:
      - api

Writing docker_lab/compose.broken.yaml


### 6.2 [실습 미션 수행] 버그 3곳을 직접 수정하고 자동 검증 스크립트 통과하기

1. 위의 `Dockerfile.broken`과 `compose.broken.yaml`에서 **버그 1, 2, 3**을 찾아 수정한 뒤 터미널에서 `docker compose -f compose.broken.yaml up -d --build`로 띄웁니다.
2. 아래 정답 확인 토글을 열기 전에 직접 수정해 보고, 마지막 검증 코드로 **4대 통과 기준(헬스체크·가중치 마운트·무인증 차단·추론 및 영구 로그 기록)**을 확인하십시오.

<details class="lms-answer">
<summary>미션 1~3 수정 정답 확인</summary>

1. **버그 1 수정 (`Dockerfile.broken`)**: `CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]`으로 변경합니다.
2. **버그 2 수정 (`compose.broken.yaml`의 `ui.environment`)**: `API_BASE_URL=http://api:8000`으로 변경합니다.
3. **버그 3 수정 (`compose.broken.yaml`의 `api.volumes`)**: `- ./models:/app/models:ro`로 변경합니다.

</details>

In [13]:
# [자체 점검] 터미널에서 docker compose up -d --build 실행 후 아래 검증 스크립트로 4대 항목을 채점합니다
import requests
from pathlib import Path

def verify_d50_serving(base_url="http://127.0.0.1:8000"):
    checks = []
    try:
        # 1. 헬스체크 및 가중치 볼륨 마운트 확인
        r_health = requests.get(f"{base_url}/health", timeout=3)
        h_data = r_health.json()
        checks.append(("1. /health 200 응답", r_health.status_code == 200, h_data))
        checks.append(("2. 모델 가중치 볼륨 마운트(weights_mounted=True)", h_data.get("weights_mounted") is True, h_data.get("model_source")))

        # 2. 무인증 요청 차단 (401) 확인
        r_unauth = requests.post(f"{base_url}/predict", json={"text": "테스트"}, timeout=3)
        checks.append(("3. 무인증 요청 401 차단", r_unauth.status_code == 401, r_unauth.status_code))

        # 3. 정상 인증 추론 (200) 및 호스트 볼륨 로그 파일 확인
        r_pred = requests.post(
            f"{base_url}/predict",
            json={"text": "컨테이너 배포가 빠르다 그리고 안정적이다"},
            headers={"X-API-Key": "modu-private-llm-key"},
            timeout=3
        )
        log_exists = Path("docker_lab/logs/serving.log").exists()
        checks.append(("4. 정상 추론 200 및 호스트 볼륨 로그 기록", r_pred.status_code == 200 and log_exists, r_pred.json()))
    except Exception as e:
        print(f"연결 대기 중 (터미널에서 docker compose up -d --build 실행 시 통과): {e}")
        return

    for name, passed, detail in checks:
        mark = "PASS" if passed else "FAIL"
        print(f"[{mark}] {name} -> {detail}")

verify_d50_serving()

[PASS] 1. /health 200 응답 -> {'status': 'healthy', 'model_version': '1.1.0-private', 'model_source': 'private-ko-sentiment-v1', 'weights_mounted': True}
[PASS] 2. 모델 가중치 볼륨 마운트(weights_mounted=True) -> private-ko-sentiment-v1
[PASS] 3. 무인증 요청 401 차단 -> 401
[PASS] 4. 정상 추론 200 및 호스트 볼륨 로그 기록 -> {'label': 'positive', 'score': 2.5, 'model_source': 'private-ko-sentiment-v1', 'model_version': '1.1.0-private'}


## 오늘 정리

- **도커 핵심 개념**: 실행 환경을 통째로 담은 읽기 전용 템플릿인 **이미지**, 그 위에 얇은 쓰기 레이어를 얹어 실행 중인 **컨테이너**, 이미지를 공유하는 **도커 허브**로 구성되며, 호스트 리눅스 커널을 공유해 가상머신(VM)보다 훨씬 가볍고 빠르게 기동됩니다.
- **설치와 기본 명령어**: Windows(WSL2 + `docker-ce`), macOS(`Colima` 또는 `Docker Desktop`), Linux 환경에서 `pull`, `images`, `run -dit -p`, `ps -a`, `logs -f`, `exec -it`, `stats`, `rm`, `rmi`, `system prune -a`로 생명주기와 자원을 관리합니다.
- **레이어 캐시·시크릿·가중치 마운트**: `COPY requirements.txt`를 코드 복사보다 앞에 두어 재빌드 속도를 높이고(`dive`로 검증), `.dockerignore`(`**/.env`)로 시크릿 유입을 막으며, 대용량 모델 가중치(`models/`)는 이미지에 굽지 않고 읽기 전용 볼륨(`-v ./models:/app/models:ro`)으로 연결하고 GPU는 `--gpus all`로 패스스루합니다.
- **Docker Compose와 트러블슈팅**: `compose.yaml`에서는 `"127.0.0.1:포트:포트"` 바인딩으로 `ufw` 방화벽 우회를 막고, 컨테이너 간에는 서비스 이름(`http://api:8000`)으로 통신합니다.

### 제출
1. 섹션 2에서 실행한 `docker ps -a`, `docker images`, `nginx` 접속 및 `exec -it` 실습 화면 캡처
2. 섹션 3~4에서 수행한 `Dockerfile.bad` vs `Dockerfile` 재빌드 시간 비교, `dive` 분석, 가중치 볼륨 마운트(`/health`의 `weights_mounted: true`) 결과 캡처
3. 섹션 5~6에서 `docker compose up -d --build`로 띄운 `api` + `ui` 실행 화면과 3대 장애 수정 후 검증 결과 캡처
4. 각 섹션 체크포인트의 답변

## Reference
- 교재 : 『LLMOps 완벽 가이드』 6.1.2 (2단계: LLM 컨테이너화하기), 6.4 (자격 증명 관리), 6.7.2 (마이크로서비스 기반 아키텍처 구현)
- Docker 공식 문서 : [Docker Engine Install](https://docs.docker.com/engine/install/), [Dockerfile Reference](https://docs.docker.com/reference/dockerfile/), [Compose File Reference](https://docs.docker.com/reference/compose-file/)
- NVIDIA Container Toolkit 문서 : [Installing the NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html)